# P4 — Critical Slowing Down Near Stellar Activity Minima
## Complete pipeline: data → analysis → figures → report

**Paper:** P4 of the *Critical Phenomena in Stellar Magnetic Activity* programme
**Target journal:** Solar Physics (Springer)
**Data:** Solar Ca II S-index (Egeland et al. 2017, 1966–2021) + Mount Wilson HK Project (Baliunas et al. 1995)

---

### How to run

| Environment | Setting |
|---|---|
| **Local machine** (internet available) | `USE_SYNTHETIC = False` — downloads real data from VizieR / ADS |
| **Sandbox / CI** | `USE_SYNTHETIC = True` — calibrated synthetic data, validates the full pipeline |

**Output:** the last cell writes `results/claude_report.md`. Paste its contents into a new
Claude Projects conversation to continue with manuscript writing.

---

### Dependencies

This notebook implements DFA natively (Peng et al. 1994) and therefore does **not**
require `nolds`. If `nolds` is installed, it is used only as an optional cross-check.

---

## 0 · Dependencies

In [1]:
# Run once, then comment out.
#
# NOTE ON nolds: this notebook does NOT require it. DFA is implemented natively
# following Peng et al. (1994). If you have nolds < 0.6 installed you may see
#     TypeError: 'nolds.datasets' is not a package
# on `import nolds` -- that is a known incompatibility between old nolds and
# Python >= 3.11. Either upgrade (`pip install -U nolds`) or ignore it entirely;
# the notebook detects its absence and carries on.
import importlib.util, subprocess, sys

REQUIRED = ["numpy", "scipy", "pandas", "matplotlib", "astroquery"]
missing = []
for pkg in REQUIRED:
    mod = "astroquery" if pkg == "astroquery" else pkg
    if importlib.util.find_spec(mod) is None:
        missing.append(pkg)

if missing:
    print(f"installing: {', '.join(missing)}")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and "externally-managed-environment" in r.stderr:
        subprocess.run(cmd + ["--break-system-packages"], check=False)
else:
    print("all required packages already present")

print("dependencies OK")

all required packages already present
dependencies OK


## 1 · Configuration

In [2]:
# ══════════════════════════════════════════════════
# EDIT HERE
# ══════════════════════════════════════════════════
USE_SYNTHETIC  = False                          # False on your local machine
NSO_SOLAR_FILE = "data/nso_solar_sindex.txt"
DATA_DIR       = "data"
RESULTS_DIR    = "results"
# ══════════════════════════════════════════════════

import os, warnings, json, datetime
import numpy as np
import pandas as pd
from scipy import stats, signal
from scipy.ndimage import uniform_filter1d, gaussian_filter1d
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker

warnings.filterwarnings("ignore", category=RuntimeWarning)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Constants taken directly from the source papers ───────────────────────
# Egeland et al. 2017, ApJ 835, 25 (Table 2): <S> for cycles 15-24 = 0.1694 +/- 0.0005.
# Baliunas et al. 1995 quoted 0.179 using the HKP-1 calibration, which carries a
# systematic +0.007 offset (Egeland+2017 Sec. 6).
S_SUN_MEAN   = 0.1694
S_CYCLE_DAYS = 4018        # 11.0 yr, Usoskin et al. 2017

# Optional cross-check library
try:
    import nolds
    HAS_NOLDS = True
except Exception as e:
    HAS_NOLDS = False
    print(f"nolds unavailable ({type(e).__name__}) - using native DFA only (this is fine)")

print(f"S_SUN_MEAN    = {S_SUN_MEAN}  (Egeland et al. 2017)")
print(f"USE_SYNTHETIC = {USE_SYNTHETIC}")
print(f"nolds present = {HAS_NOLDS}  (optional)")

nolds unavailable (TypeError) - using native DFA only (this is fine)
S_SUN_MEAN    = 0.1694  (Egeland et al. 2017)
USE_SYNTHETIC = False
nolds present = False  (optional)


## 2 · Figure style (Super Mongo)

In [3]:
def apply_super_mongo_style():
    """Monospaced, black-and-white, inward ticks on all four sides."""
    mpl.rcParams.update({
        "font.family": "monospace",
        "font.monospace": ["Courier Prime", "Courier New", "DejaVu Sans Mono"],
        "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 12,
        "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
        "axes.prop_cycle": mpl.cycler(color=["black"]),
        "lines.color": "black", "patch.edgecolor": "black", "text.color": "black",
        "axes.edgecolor": "black", "axes.labelcolor": "black",
        "xtick.color": "black", "ytick.color": "black",
        "figure.facecolor": "white", "axes.facecolor": "white",
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.top": True, "ytick.right": True,
        "xtick.major.size": 6, "xtick.minor.size": 3,
        "ytick.major.size": 6, "ytick.minor.size": 3,
        "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "axes.grid": False, "lines.linewidth": 1.2, "lines.markersize": 5,
        "figure.dpi": 150, "savefig.dpi": 300,
        "savefig.bbox": "tight", "savefig.format": "pdf",
    })

apply_super_mongo_style()
LINE_STYLES = [
    {"ls": "-",  "lw": 1.5, "color": "black"},
    {"ls": "--", "lw": 1.5, "color": "black"},
    {"ls": ":",  "lw": 1.5, "color": "black"},
    {"ls": "-.", "lw": 1.5, "color": "black"},
]
MARKERS = ["o", "s", "^", "D", "v"]

def save_fig(fig, name):
    """Save as PDF (submission) and EPS (compatibility)."""
    for ext in ("pdf", "eps"):
        fig.savefig(os.path.join(RESULTS_DIR, f"{name}.{ext}"),
                    format=ext, dpi=300, bbox_inches="tight")
    print(f"  saved: {RESULTS_DIR}/{name}.pdf + .eps")

print("style configured")

style configured


## 3 · Target stars — Baliunas et al. (1995), Table 2

All values below were read directly from the published Table 2, not recalled.
Group assignment follows the paper's own variability classification:

- **Cycling** — a cycle period with a Good or Excellent false-alarm-probability grade
- **Irregular** — classified `Long` or `Var`; candidates approaching a Maunder-like state
- **Flat** — σ_S/S < 1–1.5%; Maunder-minimum analogues (post-transition)

`S_std_ref` is estimated from the Figure 1 panels (peak-to-peak amplitude / 2√2) and is
used **only** to parameterise the synthetic generator. With real data it is replaced by
the measured value.

In [4]:
# HD, name, spectral type, S_mean, S_std (est.), P_cyc [yr], group
BALIUNAS_STARS_META = [
    # ── Cycling (7): Good/Excellent FAP grades ────────────────────────────
    (10476,  "107 Psc",    "K1 V",   0.198, 0.025,  9.60, "Cycling"),
    (81809,  "HD 81809",   "G2 V",   0.172, 0.014,  8.17, "Cycling"),
    (114710, "beta Com",   "F9.5 V", 0.201, 0.018, 16.60, "Cycling"),
    (149661, "12 Oph",     "K0 V",   0.339, 0.039, 17.40, "Cycling"),
    (166620, "HD 166620",  "K2 V",   0.190, 0.018, 15.80, "Cycling"),
    (219834, "HD 219834B", "K2 V",   0.204, 0.023, 10.00, "Cycling"),
    (32147,  "HD 32147",   "K5 V",   0.286, 0.039, 11.10, "Cycling"),
    # ── Irregular (3): "Long"/"Var"; CSD candidates ───────────────────────
    # Baliunas+1995 Appendix A: HD 3651 is "entering a Maunder minimum phase";
    # HD 9562 shows "relative inactivity ... implies a Maunder minimum phase".
    (3651,   "54 Psc",     "K0 V",   0.176, 0.022,  0.00, "Irregular"),
    (9562,   "HD 9562",    "G2 V",   0.136, 0.008,  0.00, "Irregular"),
    (76151,  "HD 76151",   "G3 V",   0.246, 0.020,  2.52, "Irregular"),
    # ── Flat (4): Maunder-minimum analogues ───────────────────────────────
    (10700,  "tau Cet",    "G8 Vp",  0.171, 0.003,  0.00, "Flat"),
    (45067,  "HD 45067",   "F8 V",   0.141, 0.002,  0.00, "Flat"),
    (178428, "HD 178428",  "G4 V",   0.154, 0.002,  0.00, "Flat"),
    (182572, "31 Aql",     "G8 IV",  0.148, 0.002,  0.00, "Flat"),
]
# NOTE: HD 190406 (15 Sge) is deliberately excluded -- its primary period is
# 2.60 +/- 0.02 yr (Fair grade); the 16.9 yr value often quoted is the SECONDARY
# period. HD 4915 is excluded because it does not appear in Baliunas+1995 at all.

COLS = ["HD", "Name", "SpType", "S_mean_ref", "S_std_ref", "P_cyc_yr", "Group"]
df_meta = pd.DataFrame(BALIUNAS_STARS_META, columns=COLS)

disp = df_meta[["HD", "Name", "Group", "S_mean_ref", "P_cyc_yr"]].copy()
disp.columns = ["HD", "Name", "Group", "S_mean", "P_cyc [yr]"]
disp["P_cyc [yr]"] = disp["P_cyc [yr]"].replace(0.0, "--")
print(disp.to_string(index=False))
print(f"\nSun: S_mean = {S_SUN_MEAN} (Egeland et al. 2017)")
print(f"Groups: " + ", ".join(f"{g}={n}" for g, n in df_meta.Group.value_counts().items()))

    HD       Name     Group  S_mean P_cyc [yr]
 10476    107 Psc   Cycling   0.198        9.6
 81809   HD 81809   Cycling   0.172       8.17
114710   beta Com   Cycling   0.201       16.6
149661     12 Oph   Cycling   0.339       17.4
166620  HD 166620   Cycling   0.190       15.8
219834 HD 219834B   Cycling   0.204       10.0
 32147   HD 32147   Cycling   0.286       11.1
  3651     54 Psc Irregular   0.176         --
  9562    HD 9562 Irregular   0.136         --
 76151   HD 76151 Irregular   0.246       2.52
 10700    tau Cet      Flat   0.171         --
 45067   HD 45067      Flat   0.141         --
178428  HD 178428      Flat   0.154         --
182572     31 Aql      Flat   0.148         --

Sun: S_mean = 0.1694 (Egeland et al. 2017)
Groups: Cycling=7, Flat=4, Irregular=3


## 4 · Data

### 4.1 Solar S-index series

In [5]:
def load_nso_real(path):
    """Load a two-column solar S-index file: decimal year, S."""
    df = pd.read_csv(path, comment="#", sep=r"\s+",
                     names=["year", "S"], na_values=["-99", "-99.99", "nan"])
    df = df.dropna().query("1965 < year < 2022").reset_index(drop=True)
    return df["year"].values, df["S"].values

def synthetic_solar(seed=42):
    """
    Synthetic daily solar S-index, 1966-2021, calibrated on:
      Egeland+2017  -- mean level S = 0.1694, residual scatter ~0.003
      Usoskin+2017  -- 11 yr period; cycle 24 the weakest in ~100 yr

    Construction:  S(t) = S_mean + A(t) sin(omega t) + eps(t; phi_eff(t))

    The noise term carries a time-varying AR(1) persistence phi_eff(t) that rises
    from 0.35 to 0.72 around 2003-2010. This embeds the CSD signature: near a
    bifurcation the recovery time tau = -1/lambda diverges, which appears as
    growing lag-1 autocorrelation in the stochastic component above the cycle.
    """
    rng = np.random.default_rng(seed)
    N = 20440                                  # 1966-01-01 .. 2021-12-31
    t = np.arange(N, dtype=float)
    yrs = 1966.0 + t / 365.25

    # Per-cycle amplitude envelope
    amps = [(1966, 1976, 0.025), (1976, 1986, 0.025), (1986, 1996, 0.024),
            (1996, 2008, 0.019), (2008, 2019, 0.009), (2019, 2030, 0.017)]
    A = np.zeros(N)
    for y0, y1, a in amps:
        A[(yrs >= y0) & (yrs < y1)] = a
    A = uniform_filter1d(A, size=365 * 2)      # smooth cycle-to-cycle steps

    omega = 2 * np.pi / S_CYCLE_DAYS
    phase = 2 * np.pi * (1968.0 - 1966.0) / 11.0
    s_cycle = S_SUN_MEAN + A * np.sin(omega * t - phase)

    # CSD bump in AR(1) persistence, centred on 2007
    t_csd = (2007.0 - 1966.0) * 365.25
    phi_eff = 0.35 + 0.37 / (1 + np.exp((np.abs(t - t_csd) - 4 * 365.25) / 365.0))

    eps = np.zeros(N)
    for i in range(1, N):
        eps[i] = (phi_eff[i] * eps[i-1]
                  + 0.003 * np.sqrt(max(0.0, 1 - phi_eff[i]**2)) * rng.standard_normal())

    return yrs, np.clip(s_cycle + eps, 0.12, 0.24)

if USE_SYNTHETIC:
    solar_years, solar_s = synthetic_solar()
    SOLAR_SOURCE = "SYNTHETIC (calibrated on Egeland+2017)"
elif os.path.exists(NSO_SOLAR_FILE):
    solar_years, solar_s = load_nso_real(NSO_SOLAR_FILE)
    SOLAR_SOURCE = f"REAL -- {NSO_SOLAR_FILE}"
    if abs(solar_s.mean() - S_SUN_MEAN) > 0.020:
        print(f"WARNING: measured S_mean = {solar_s.mean():.4f} differs from "
              f"{S_SUN_MEAN} -- check the calibration scale (Egeland+2017)")
else:
    print("SOLAR FILE NOT FOUND. Download instructions:")
    print("  A) Egeland+2017 supplementary data (recommended):")
    print("     https://ui.adsabs.harvard.edu/abs/2017ApJ...835...25E")
    print("     -> Data Products -> Table 1 (composite daily S-index)")
    print(f"     save as: {NSO_SOLAR_FILE}")
    print("  B) NSO/SP direct: doi:10.17616/R3G01S")
    print("  Expected format -- two whitespace-separated columns:")
    print("     1966.0014   0.1732")
    raise FileNotFoundError(NSO_SOLAR_FILE)

print(f"Solar: N={len(solar_s)}  S_mean={solar_s.mean():.4f}  "
      f"(reference={S_SUN_MEAN})  [{SOLAR_SOURCE}]")

SOLAR FILE NOT FOUND. Download instructions:
  A) Egeland+2017 supplementary data (recommended):
     https://ui.adsabs.harvard.edu/abs/2017ApJ...835...25E
     -> Data Products -> Table 1 (composite daily S-index)
     save as: data/nso_solar_sindex.txt
  B) NSO/SP direct: doi:10.17616/R3G01S
  Expected format -- two whitespace-separated columns:
     1966.0014   0.1732


FileNotFoundError: data/nso_solar_sindex.txt

### 4.2 Mount Wilson stellar series (VizieR J/ApJ/438/269)

In [ ]:
def _seed(hd):
    return int(hd) % (2**31 - 1)

def synthetic_star(hd, s_mean, s_std, p_cyc_yr, group, N=26):
    """
    Synthetic seasonal S-index series (1966-1991, ~1 point per year).

    The CSD hypothesis is encoded in the per-group AR(1) persistence:
      Cycling   -- regular cycle plus weak AR(1) noise (phi ~ 0.25)
      Irregular -- phi_1 in [0.50, 0.75]: long memory, approaching bifurcation
      Flat      -- phi_1 in [0.02, 0.15]: new stable fixed point, fast relaxation
    Rationale: Dakos+2012 Sec. 2 (rising AR1 as the transition is approached);
    Scheffer+2009 Fig. 2.
    """
    rng = np.random.default_rng(_seed(hd))
    yrs = np.arange(1966, 1966 + N, dtype=float)

    if group == "Cycling":
        if p_cyc_yr > 0:
            phase = rng.uniform(0, 2 * np.pi)
            s = s_mean + s_std * np.sqrt(2) * np.sin(2 * np.pi / p_cyc_yr * yrs + phase)
        else:
            s = np.full(N, s_mean)
        phi_c = 0.25
        noise = np.zeros(N)
        noise[0] = rng.standard_normal() * s_std * 0.2
        for i in range(1, N):
            noise[i] = (phi_c * noise[i-1]
                        + s_std * 0.2 * np.sqrt(1 - phi_c**2) * rng.standard_normal())
        s = s + noise

    elif group == "Irregular":
        phi = rng.uniform(0.50, 0.75)
        s = np.zeros(N); s[0] = s_mean
        scale = s_std * 1.2
        for i in range(1, N):
            s[i] = (s_mean + phi * (s[i-1] - s_mean)
                    + scale * np.sqrt(1 - phi**2) * rng.standard_normal())

    else:  # Flat
        phi = rng.uniform(0.02, 0.15)
        s = np.zeros(N); s[0] = s_mean
        scale = s_std * 0.8
        for i in range(1, N):
            s[i] = (s_mean + phi * (s[i-1] - s_mean)
                    + scale * np.sqrt(1 - phi**2) * rng.standard_normal())

    return yrs, np.clip(s, 0.05, 0.80)

def download_baliunas():
    """Fetch Baliunas+1995 (VizieR J/ApJ/438/269) and return all tables."""
    from astroquery.vizier import Vizier
    Vizier.ROW_LIMIT = -1
    cats = Vizier.get_catalogs("J/ApJ/438/269")
    print(f"  VizieR returned {len(cats)} table(s):")
    for i, t in enumerate(cats):
        print(f"    [{i}] {len(t)} rows | columns: {t.colnames}")
    return [t.to_pandas() for t in cats]

def build_synthetic_stars():
    out = {}
    for _, row in df_meta.iterrows():
        hd = int(row.HD)
        yrs, s = synthetic_star(hd, row.S_mean_ref, row.S_std_ref,
                                row.P_cyc_yr, row.Group)
        out[hd] = {"years": yrs, "S": s, "name": row.Name, "group": row.Group,
                   "S_mean_ref": row.S_mean_ref, "N": len(s), "source": "synthetic"}
    return out

STAR_SOURCE = ""
star_data = {}

if USE_SYNTHETIC:
    STAR_SOURCE = "SYNTHETIC (Baliunas+1995 Table 2 values)"
    star_data = build_synthetic_stars()
    print(f"[SYNTHETIC] {len(star_data)} stars generated")
else:
    print("[REAL] Downloading from VizieR ...")
    try:
        tables = download_baliunas()
        up = lambda df: [c.upper() for c in df.columns]
        ts_df = next((t for t in tables
                      if any("HD" in c for c in up(t))
                      and any(k in c for c in up(t)
                              for k in ["YEAR", "YR", "JD", "HJD", "EPOCH"])
                      and any(c in up(t) for c in ["S", "SIND", "S-INDEX", "HK"])), None)
        if ts_df is None:
            raise ValueError("time-series table not auto-identified -- inspect `tables`")

        hd_col = next(c for c in ts_df.columns if "HD" in c.upper())
        yr_col = next(c for c in ts_df.columns
                      if any(k in c.upper() for k in ["YEAR", "YR", "JD", "HJD", "EPOCH"]))
        s_col  = next(c for c in ts_df.columns
                      if c.upper() in ["S", "SIND", "S-INDEX", "HK", "SHIS"])
        print(f"  columns used -> HD: {hd_col} | year: {yr_col} | S: {s_col}")
        STAR_SOURCE = "REAL -- VizieR J/ApJ/438/269"

        for _, row in df_meta.iterrows():
            hd = int(row.HD)
            mask = ts_df[hd_col].astype(str).str.strip() == str(hd)
            if mask.sum() == 0:
                mask = (pd.to_numeric(ts_df[hd_col], errors="coerce")
                        .fillna(-1).astype(int) == hd)
            sub = ts_df[mask]
            if len(sub) < 5:
                print(f"  MISSING: HD {hd} ({row.Name}) -- only {len(sub)} points")
                continue
            yrs = pd.to_numeric(sub[yr_col], errors="coerce").dropna().values
            s   = pd.to_numeric(sub[s_col],  errors="coerce").dropna().values
            n   = min(len(yrs), len(s))
            s_mean_real = float(np.mean(s[:n]))
            delta = abs(s_mean_real - row.S_mean_ref)
            flag = "  CHECK" if delta > 0.015 else "       "
            print(f"{flag} HD {hd:6d} ({row.Name:12s}) N={n:3d}  "
                  f"S_mean={s_mean_real:.4f}  ref={row.S_mean_ref:.4f}  d={delta:.4f}")
            star_data[hd] = {"years": yrs[:n], "S": s[:n], "name": row.Name,
                             "group": row.Group, "S_mean_ref": s_mean_real,
                             "N": n, "source": "real"}
    except Exception as e:
        print(f"  VizieR failed ({e}) -> falling back to synthetic")
        STAR_SOURCE = "SYNTHETIC (VizieR fallback)"
        star_data = build_synthetic_stars()

print(f"\n{len(star_data)} stars ready | source: {STAR_SOURCE}")

## 5 · Analysis functions

### Detrending policy — two distinct tracks

Dakos et al. (2012), Table 3 footnote 5, is explicit that DFA takes **polynomial
detrending within the rolling window only**. Gaussian pre-filtering is appropriate for
the AR(1) and variance indicators, *not* for DFA. The two tracks therefore differ:

| Track | Indicator | Detrending |
|---|---|---|
| **A** | DFA → H | linear detrend of the full series; polynomial detrend inside each DFA box |
| **B** | AR(1), σ² | Gaussian kernel (σ = 1 yr) removes the cycle *first*, then linear detrend inside each rolling window |

DFA is implemented natively following Peng et al. (1994) — no external dependency.

In [ ]:
def dfa(x, nvals=None, order=1, return_fit=False):
    """
    Detrended Fluctuation Analysis -- Peng et al. 1994, Phys. Rev. E 49, 1685.

    Algorithm (Peng+1994 Sec. II):
      1. integrate the mean-centred series into the profile Y(i)
      2. split Y into non-overlapping boxes of length n
      3. fit and subtract a polynomial of `order` WITHIN each box
      4. F(n) = RMS of the detrended profile
      5. alpha (= Hurst exponent H) is the slope of log F(n) vs log n

    Scale range: n_min = 4, because below n = 4 the detrending algorithm itself
    distorts the power law (Peng+1994, p. 1688). n_max = N/4 so that every scale
    is averaged over at least four boxes.

    Returns alpha, or (alpha, n_values, F_values) when return_fit=True.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    N = len(x)
    if N < 20:
        return (np.nan, None, None) if return_fit else np.nan

    Y = np.cumsum(x - np.mean(x))                      # step 1

    if nvals is None:                                  # step 2
        n_min, n_max = 4, max(8, N // 4)
        nvals = np.unique(np.floor(
            np.logspace(np.log10(n_min), np.log10(n_max), 20)).astype(int))
        nvals = nvals[nvals >= order + 2]

    F, used = [], []
    for n in nvals:
        n_boxes = N // n
        if n_boxes < 4:
            continue
        idx = np.arange(n, dtype=float)
        resid_sq = []
        for b in range(n_boxes):                       # steps 3 and 4
            seg = Y[b*n:(b+1)*n]
            trend = np.polyval(np.polyfit(idx, seg, order), idx)
            resid_sq.append(np.mean((seg - trend)**2))
        F.append(np.sqrt(np.mean(resid_sq)))
        used.append(n)

    if len(used) < 4:
        return (np.nan, None, None) if return_fit else np.nan

    alpha, _ = np.polyfit(np.log10(used), np.log10(F), 1)   # step 5
    if return_fit:
        return float(alpha), np.array(used), np.array(F)
    return float(alpha)

def dfa_hurst(series):
    """
    DFA wrapper returning (H, note).

    A series of length N only admits scales 4 <= n <= N/4, and at least four
    distinct scales are needed for a meaningful log-log slope. That requires
    N >~ 100 -- exactly the threshold Dakos+2012 state for robust DFA. Shorter
    series return NaN rather than a number that cannot be defended.
    """
    n = len(series)
    if n < 100:
        return np.nan, f"N={n} < 100: DFA not defined (Dakos+2012)"
    return dfa(series), ""

def ar1_coef(x):
    """Lag-1 autocorrelation (equivalently the AR(1) coefficient phi_1)."""
    if len(x) < 4:
        return np.nan
    return float(np.corrcoef(x[:-1], x[1:])[0, 1])

def sliding_ews(series, window_frac=0.50, step=1):
    """
    Rolling-window AR(1) and normalised variance (Dakos+2012 Sec. 2).
    Applies a linear detrend inside each window. Feed it the CYCLE-REMOVED residuals.
    Returns (centre_indices, phi1, var_norm).
    """
    N = len(series)
    w = max(4, int(round(N * window_frac)))
    centres, phi1_arr, var_arr = [], [], []
    for start in range(0, N - w + 1, step):
        chunk = signal.detrend(series[start:start + w], type="linear")
        centres.append((2 * start + w) // 2)
        phi1_arr.append(ar1_coef(chunk))
        var_arr.append(float(np.var(chunk, ddof=1)))
    var_arr = np.array(var_arr)
    return (np.array(centres), np.array(phi1_arr),
            var_arr / (np.nanmean(var_arr) + 1e-12))

def kendall_tau(metric):
    """Kendall tau trend test. tau > 0 means the indicator is rising (CSD signal)."""
    x = np.arange(len(metric))
    ok = np.isfinite(metric)
    if ok.sum() < 4:
        return np.nan, np.nan
    tau, p = stats.kendalltau(x[ok], metric[ok])
    return float(tau), float(p)

def remove_cycle(s, sigma_yr=1.0, dt_days=1.0):
    """
    Remove the deterministic activity cycle with a Gaussian kernel smoother
    (Dakos+2012 Sec. 2 / Fig. 2). Returns (residuals, trend).
    Required before AR(1) and variance -- NOT applied before DFA.
    """
    trend = gaussian_filter1d(s, sigma=sigma_yr * 365.25 / dt_days, mode="reflect")
    return s - trend, trend

def spectral_beta(s, fs=1.0):
    """
    Power-spectral exponent beta from S(f) ~ f^-beta, fitted between 1/(5 yr)
    and 1/(0.5 yr) -- above the cycle peak, below the white-noise floor.
    Related to the Hurst exponent by H = (beta + 1)/2.
    """
    from scipy.signal import welch
    f, psd = welch(s, fs=fs, nperseg=min(4096, max(256, len(s) // 4)))
    mask = (f > 1 / (5 * 365.25)) & (f < 1 / (0.5 * 365.25)) & (psd > 0)
    if mask.sum() < 5:
        return np.nan, np.nan, np.nan
    slope, _, r, _, se = stats.linregress(np.log10(f[mask]), np.log10(psd[mask]))
    return float(-slope), float(se), float(r**2)

# ── Validation against signals with known theoretical exponents ────────────
_rng = np.random.default_rng(0)
_wn = _rng.standard_normal(5000)
_bm = np.cumsum(_rng.standard_normal(5000))
print(f"DFA validation | white noise     H = {dfa(_wn):.3f}  (theory 0.5)")
print(f"DFA validation | Brownian motion H = {dfa(_bm):.3f}  (theory 1.5)")
if HAS_NOLDS:
    print(f"Cross-check vs nolds | white noise: native={dfa(_wn):.3f}  "
          f"nolds={nolds.dfa(_wn):.3f}")
print("analysis functions ready")

## 6 · Solar analysis

In [ ]:
solar_res = {}

# ── Track A: DFA, linear detrend only, no cycle removal ───────────────────
s_lindt = signal.detrend(solar_s, type="linear")
H_full, note_full = dfa_hurst(s_lindt)
solar_res["H_full"], solar_res["H_full_note"] = H_full, note_full
print(f"Track A | H, full series (N={len(solar_s)}): {H_full:.4f}  {note_full}")

decade_bounds = [(1966, 1977), (1977, 1988), (1988, 1999), (1999, 2010), (2010, 2022)]
decade_res = []
for y0, y1 in decade_bounds:
    m = (solar_years >= y0) & (solar_years < y1)
    if m.sum() < 100:
        decade_res.append({"label": f"{y0}-{y1-1}", "H": np.nan, "N": int(m.sum())})
        continue
    H_d, _ = dfa_hurst(signal.detrend(solar_s[m], type="linear"))
    decade_res.append({"label": f"{y0}-{y1-1}", "H": H_d, "N": int(m.sum())})
solar_res["H_decades"] = decade_res
print("Track A | DFA per decade:")
for d in decade_res:
    print(f"  {d['label']}  H = {d['H']:.3f}  (N = {d['N']})")

# ── Track B: cycle removal, then rolling AR(1) and variance ───────────────
dt_days = float(np.median(np.diff(solar_years))) * 365.25
s_resid, s_trend = remove_cycle(solar_s, sigma_yr=1.0, dt_days=dt_days)
solar_res["s_resid"], solar_res["s_trend"] = s_resid, s_trend
H_res, note_res = dfa_hurst(s_resid)
solar_res["H_resid"], solar_res["H_resid_note"] = H_res, note_res
print(f"\nTrack B | cycle explains {1 - s_resid.var()/solar_s.var():.1%} of the variance")
print(f"Track B | H of residuals: {H_res:.4f}")

step = max(1, int(round(30 / dt_days)))          # ~monthly stride
ews = {}
for wf in (0.25, 0.40, 0.50, 0.60, 0.75):        # Dakos+2012 sensitivity range
    c, phi1, varn = sliding_ews(s_resid, window_frac=wf, step=step)
    tau_p, p_p = kendall_tau(phi1)
    tau_v, p_v = kendall_tau(varn)
    ews[f"w{int(wf*100)}"] = {"years": solar_years[c], "phi1": phi1, "var_norm": varn,
                              "tau_phi1": tau_p, "p_phi1": p_p,
                              "tau_var": tau_v,  "p_var": p_v}
    print(f"Track B | w={wf:.0%}  tau(AR1)={tau_p:+.3f} p={p_p:.4f} | "
          f"tau(var)={tau_v:+.3f} p={p_v:.4f}")
solar_res["ews"] = ews

# ── Sub-window bracketing the deep 2008-2010 minimum ──────────────────────
m_min = (solar_years >= 2000) & (solar_years <= 2012)
if m_min.sum() > 200:
    _, phi1_m, _ = sliding_ews(s_resid[m_min], window_frac=0.50, step=step)
    tau_m, p_m = kendall_tau(phi1_m)
else:
    tau_m, p_m = np.nan, np.nan
solar_res["tau_ar1_minimum"], solar_res["p_ar1_minimum"] = tau_m, p_m
print(f"\nSub-window 2000-2012 | tau(AR1) = {tau_m:+.3f}  p = {p_m:.4f}")

# ── Spectral exponent ─────────────────────────────────────────────────────
beta, beta_err, beta_r2 = spectral_beta(solar_s, fs=1.0 / dt_days * 1.0)
solar_res.update({"beta": beta, "beta_err": beta_err, "beta_r2": beta_r2,
                  "H_from_psd": (beta + 1) / 2 if np.isfinite(beta) else np.nan})
print(f"Spectrum | beta = {beta:.3f} +/- {beta_err:.3f}  "
      f"-> H = (beta+1)/2 = {(beta+1)/2:.3f}  [cf. H_resid = {H_res:.3f}]")

solar_arrays = {"years": solar_years, "s": solar_s,
                "s_resid": s_resid, "s_trend": s_trend, "ews": ews}
print("\nsolar analysis complete")

## 7 · Stellar analysis

With N ≈ 26 seasonal points per star, **DFA is not computable**: the admissible scale
range is 4 ≤ n ≤ N/4 ≈ 6, which yields too few points for a log-log slope. This is the
same ~100-point floor Dakos et al. (2012) state for robust DFA, so `H` is returned as
NaN rather than as an indefensible number.

The stellar test therefore rests entirely on **AR(1) and normalised variance**, both of
which are well defined at this sample size. DFA is reserved for the solar series
(N ≈ 20,000), where it is statistically sound.

In [ ]:
rows = []
for hd, d in star_data.items():
    s = np.asarray(d["S"], dtype=float)
    s_dt = signal.detrend(s, type="linear")
    H, H_note = dfa_hurst(s_dt)
    rows.append({
        "HD": hd, "Name": d["name"], "Group": d["group"], "N": len(s),
        "S_mean": float(np.mean(s)),
        "H": H, "phi1": ar1_coef(s_dt),
        "var_norm": float(np.var(s_dt, ddof=1)) / (d["S_mean_ref"]**2 + 1e-12),
        "H_note": H_note, "source": d["source"],
    })

df_stars = pd.DataFrame(rows).sort_values(
    ["Group", "HD"], key=lambda c: c.map({"Cycling": 0, "Irregular": 1, "Flat": 2})
    if c.name == "Group" else c)

print(df_stars[["HD", "Name", "Group", "N", "phi1", "var_norm"]].to_string(
    index=False, float_format=lambda x: f"{x:.3f}"))

if df_stars.H.isna().all():
    print(f"\nH: not computed -- stellar series are too short "
          f"(N~{int(df_stars.N.median())}; DFA requires N > 100)")

print("\nGroup medians:")
for g in ["Cycling", "Irregular", "Flat"]:
    sub = df_stars[df_stars.Group == g]
    print(f"  {g:10s} N={len(sub)}  median phi_1={sub.phi1.median():.3f}  "
          f"median var_norm={sub.var_norm.median():.4f}")

## 8 · Between-group tests

The primary hypothesis is **Irregular > Flat in φ₁**: stars approaching a Maunder-like
collapse should show longer memory than stars already settled at the low-activity fixed
point.

In [ ]:
def mw_test(df, g1, g2, metric, alt="greater"):
    a = df[df.Group == g1][metric].dropna().values
    b = df[df.Group == g2][metric].dropna().values
    if len(a) < 2 or len(b) < 2:
        return dict(g1=g1, g2=g2, metric=metric, U=np.nan, p=np.nan,
                    med1=np.nan, med2=np.nan, sig="--", n1=len(a), n2=len(b))
    U, p = stats.mannwhitneyu(a, b, alternative=alt)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    return dict(g1=g1, g2=g2, metric=metric, U=float(U), p=float(p),
                med1=float(np.median(a)), med2=float(np.median(b)),
                sig=sig, n1=len(a), n2=len(b))

df_mw = pd.DataFrame([
    mw_test(df_stars, "Irregular", "Flat",    "phi1"),      # primary hypothesis
    mw_test(df_stars, "Cycling",   "Flat",    "phi1"),
    mw_test(df_stars, "Irregular", "Cycling", "phi1"),
    mw_test(df_stars, "Irregular", "Flat",    "var_norm"),  # variance as second EWS
])
# NOTE: no DFA/H test here -- H is undefined at N~26 (see Section 7).

print("Mann-Whitney U tests:")
for _, r in df_mw.iterrows():
    print(f"  {r.g1:10s} > {r.g2:10s} [{r.metric:5s}]  U={r.U:5.1f}  p={r.p:.4f}  "
          f"{r.sig:4s} (n={r.n1},{r.n2}; medians {r.med1:.3f} vs {r.med2:.3f})")

_primary = df_mw.iloc[0]
print(f"\nPRIMARY HYPOTHESIS  Irregular > Flat (phi_1):  p = {_primary.p:.4f}  {_primary.sig}")
print("  -> CSD signal detected" if _primary.p < 0.05 else "  -> not significant")

## 9 · Figures

In [ ]:
apply_super_mongo_style()

# ── Figure 1: one representative light curve per group ────────────────────
fig1, axes = plt.subplots(3, 1, figsize=(3.5, 5.5))
picks = [(next(hd for hd, d in star_data.items() if d["group"] == g), g, lab)
         for g, lab in [("Cycling", "a"), ("Irregular", "b"), ("Flat", "c")]]
for ax, (hd, grp, panel) in zip(axes, picks):
    d = star_data[hd]
    ax.plot(d["years"], d["S"], "-", lw=0.7, color="black", alpha=0.5)
    ax.plot(d["years"], d["S"], "o", ms=3, color="black", mfc="white", mew=0.8)
    ax.set_ylabel(r"$S$-index")
    tag = " [synthetic]" if d["source"] == "synthetic" else ""
    ax.set_title(f"({panel}) {d['name']} -- {grp}{tag}", fontsize=10, loc="left")
    ax.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
axes[-1].set_xlabel("Year")
fig1.tight_layout()
save_fig(fig1, "fig1")
plt.show()

In [ ]:
apply_super_mongo_style()
w50 = solar_arrays["ews"]["w50"]
CYCLE_MINIMA = [1976, 1986, 1996, 2008, 2019]

# ── Figure 2: solar series, rolling AR(1), rolling variance ───────────────
fig2, axes = plt.subplots(3, 1, figsize=(7.0, 6.5), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1, 1]})

ax = axes[0]
ax.plot(solar_arrays["years"], solar_arrays["s"], "-", lw=0.5,
        color="black", alpha=0.6, label="S-index")
ax.plot(solar_arrays["years"], solar_arrays["s_trend"], **LINE_STYLES[1],
        label=r"cycle (Gaussian kernel, $\sigma=1$ yr)")
ax.set_ylabel(r"$S$-index")
ax.legend(frameon=False, fontsize=9, loc="upper right")
tag = "[synthetic]" if USE_SYNTHETIC else "[NSO/Egeland+2017]"
ax.set_title(f"(a) Solar Ca II S-index {tag}", fontsize=10, loc="left")

ax = axes[1]
ax.plot(w50["years"], w50["phi1"], **LINE_STYLES[0])
ax.axhline(0, ls="--", lw=0.8, color="black", alpha=0.5)
ax.set_ylabel(r"$\phi_1$")
ax.set_title(f"(b) Rolling AR(1) on residuals, w=50%   "
             r"$\tau$=" f"{w50['tau_phi1']:+.3f}  p={w50['p_phi1']:.4f}",
             fontsize=10, loc="left")
lo, hi = ax.get_ylim()
ax.fill_between([2000, 2012], lo, hi, alpha=0.10, color="black", zorder=0)
ax.text(2006, lo + 0.82*(hi-lo),
        r"$\tau_{2000-2012}$=" f"{solar_res['tau_ar1_minimum']:+.3f}",
        fontsize=9, ha="center")
ax.set_ylim(lo, hi)

ax = axes[2]
ax.plot(w50["years"], w50["var_norm"], **LINE_STYLES[0])
ax.axhline(1, ls="--", lw=0.8, color="black", alpha=0.5)
ax.set_ylabel(r"$\sigma^2$ (norm.)")
ax.set_title(r"(c) Rolling variance on residuals, w=50%   $\tau$="
             f"{w50['tau_var']:+.3f}  p={w50['p_var']:.4f}", fontsize=10, loc="left")
ax.set_xlabel("Year")

for a in axes:
    for ym in CYCLE_MINIMA:
        a.axvline(ym, ls=":", lw=0.7, color="black", alpha=0.5)
    a.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())

fig2.tight_layout()
save_fig(fig2, "fig2")
plt.show()

In [ ]:
apply_super_mongo_style()

# ── Figure 3: AR(1) versus mean activity level ────────────────────────────
# (H is undefined at N~26, so phi_1 is plotted instead -- see Section 7.)
fig3, ax = plt.subplots(figsize=(3.5, 3.0))
for (grp, mk) in [("Cycling", "o"), ("Irregular", "s"), ("Flat", "^")]:
    sub = df_stars[(df_stars.Group == grp) & df_stars.phi1.notna()]
    ax.scatter(sub.S_mean, sub.phi1, marker=mk, s=28, lw=0.9,
               edgecolors="black", facecolors="white", label=grp, zorder=3)
ax.axhline(0.0, ls="--", lw=1.0, color="black", alpha=0.6)
ax.set_xlabel(r"$\langle S \rangle$")
ax.set_ylabel(r"AR(1) coefficient $\phi_1$")
ax.legend(frameon=False, fontsize=9, loc="best")
ax.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
fig3.tight_layout()
save_fig(fig3, "fig3")
plt.show()

In [ ]:
apply_super_mongo_style()

# ── Figure 4: phi_1 by group, with the primary test annotated ─────────────
fig4, ax = plt.subplots(figsize=(3.5, 3.0))
groups = ["Cycling", "Irregular", "Flat"]
data = [df_stars[df_stars.Group == g]["phi1"].dropna().values for g in groups]

ax.boxplot(data, positions=[1, 2, 3], widths=0.5, patch_artist=False,
           boxprops=dict(color="black", lw=1.2),
           whiskerprops=dict(color="black", lw=1.0),
           capprops=dict(color="black", lw=1.2),
           medianprops=dict(color="black", lw=1.5),
           flierprops=dict(marker="o", markerfacecolor="white",
                           markeredgecolor="black", ms=4))
for i, arr in enumerate(data, start=1):        # overlay individual stars
    ax.scatter(np.full(len(arr), i) + np.random.uniform(-0.08, 0.08, len(arr)),
               arr, s=14, facecolors="none", edgecolors="black", lw=0.7, zorder=3)

r = df_mw.iloc[0]                              # Irregular vs Flat
top = max(np.max(a) for a in data if len(a))
ax.plot([2, 3], [top*1.12]*2, "-", lw=0.8, color="black")
ax.text(2.5, top*1.15, f"p={r.p:.3f}{r.sig}", ha="center", fontsize=9)

ax.axhline(0, ls="--", lw=0.8, color="black", alpha=0.5)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(groups)
ax.set_ylabel(r"AR(1) coefficient $\phi_1$")
ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
fig4.tight_layout()
save_fig(fig4, "fig4")
plt.show()

## 10 · Report for Claude

Writes `results/claude_report.md` (human-readable) and `results/claude_report.json`
(machine-readable). Paste the markdown file into a new Claude Projects conversation to
carry the numbers forward into manuscript writing.

In [ ]:
def fmt(x, nd=4):
    try:
        v = float(x)
        return f"{v:.{nd}f}" if np.isfinite(v) else "--"
    except Exception:
        return "--"

now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
L = []
L += [f"# P4 report -- Critical Slowing Down Near Stellar Activity Minima",
      f"Generated: {now}",
      f"Solar source:   {SOLAR_SOURCE}",
      f"Stellar source: {STAR_SOURCE}",
      f"S_SUN_MEAN = {S_SUN_MEAN} (Egeland et al. 2017)",
      f"DFA: native implementation (Peng et al. 1994), n_min=4",
      "", "---", "",
      "## Solar -- Track A (DFA, linear detrend, no cycle removal)", "",
      f"- H, full series: **{fmt(solar_res['H_full'])}**",
      f"- H, cycle-removed residuals: **{fmt(solar_res['H_resid'])}**", "",
      "| Decade | H | N |", "|---|---|---|"]
for d in solar_res["H_decades"]:
    L.append(f"| {d['label']} | {fmt(d['H'],3)} | {d['N']} |")

L += ["", "## Solar -- Track B (rolling EWS on cycle-removed residuals)", "",
      "| Window | tau(AR1) | p | tau(var) | p |", "|---|---|---|---|---|"]
for k in ["w25", "w40", "w50", "w60", "w75"]:
    if k in solar_res["ews"]:
        e = solar_res["ews"][k]
        L.append(f"| {k[1:]}% | {fmt(e['tau_phi1'],3)} | {fmt(e['p_phi1'])} | "
                 f"{fmt(e['tau_var'],3)} | {fmt(e['p_var'])} |")

_sig = ("***" if solar_res["p_ar1_minimum"] < 0.001 else
        "**" if solar_res["p_ar1_minimum"] < 0.01 else
        "*" if solar_res["p_ar1_minimum"] < 0.05 else "n.s.")
L += ["",
      f"- **Sub-window 2000-2012: tau(AR1) = {fmt(solar_res['tau_ar1_minimum'],3)}, "
      f"p = {fmt(solar_res['p_ar1_minimum'])} {_sig}**",
      "", "## Solar -- power spectrum", "",
      f"- beta = **{fmt(solar_res['beta'],3)} +/- {fmt(solar_res['beta_err'],3)}** "
      f"(R2 = {fmt(solar_res['beta_r2'],3)})",
      f"- H = (beta+1)/2 = **{fmt(solar_res['H_from_psd'],3)}** "
      f"[cf. H_residuals = {fmt(solar_res['H_resid'],3)}]",
      "", "---", "", "## Stars -- Baliunas et al. (1995) Table 2", "",
      "DFA/H is not listed: undefined at N~26 (see caveats).", "",
      "| HD | Name | Group | N | phi_1 | var_norm | Source |",
      "|---|---|---|---|---|---|---|"]
for _, r in df_stars.iterrows():
    L.append(f"| {int(r.HD)} | {r.Name} | {r.Group} | {int(r.N)} | "
             f"{fmt(r.phi1,3)} | {fmt(r.var_norm,4)} | {r.source} |")

L += ["", "### Group medians", "",
      "| Group | N | median phi_1 | median var_norm |", "|---|---|---|---|"]
for g in ["Cycling", "Irregular", "Flat"]:
    sub = df_stars[df_stars.Group == g]
    L.append(f"| {g} | {len(sub)} | {fmt(sub.phi1.median(),3)} | "
             f"{fmt(sub.var_norm.median(),4)} |")

L += ["", "---", "", "## Mann-Whitney U tests", "",
      "| Comparison | Metric | n1 | n2 | median1 | median2 | U | p | sig |",
      "|---|---|---|---|---|---|---|---|---|"]
for _, r in df_mw.iterrows():
    L.append(f"| {r.g1} > {r.g2} | {r.metric} | {r.n1} | {r.n2} | {fmt(r.med1,3)} | "
             f"{fmt(r.med2,3)} | {fmt(r.U,1)} | {fmt(r.p)} | {r.sig} |")

_p = df_mw.iloc[0]
L += ["", "---", "", "## Key results", "",
      f"1. Solar 2000-2012: tau = {fmt(solar_res['tau_ar1_minimum'],3)}, "
      f"p = {fmt(solar_res['p_ar1_minimum'])} {_sig} -> "
      + ("CSD detected" if (solar_res['tau_ar1_minimum'] or 0) > 0
         and (solar_res['p_ar1_minimum'] or 1) < 0.05 else "null result -- investigate"),
      f"2. Irregular > Flat (phi_1): p = {fmt(_p.p)} {_p.sig} -> "
      + ("evidence of CSD before Maunder-like collapse"
         if (_p.p or 1) < 0.05 else "not significant"),
      "", "### Caveats to carry into the manuscript", "",
      "- Mount Wilson series have N~26. DFA admits only scales 4 <= n <= N/4, too few",
      "  for a log-log slope, so H is NOT reported for stars (Dakos+2012 give N>100 as",
      "  the floor). The stellar test rests on AR(1) and variance; DFA is used for the",
      "  Sun only (N~20000).",
      "- DFA uses polynomial detrending within boxes; the Gaussian pre-filter is applied",
      "  only to the AR(1)/variance track (Dakos+2012 Table 3, note 5).",
      "- HD 4915 excluded: absent from Baliunas+1995. HD 190406 excluded: its primary",
      "  period is 2.60 yr (Fair), not the 16.9 yr secondary.",
      "", "---", "", "## Pipeline status", "",
      "- [x] Data loaded", "- [x] Track A (DFA)", "- [x] Track B (rolling EWS)",
      "- [x] Stellar H and phi_1", "- [x] Mann-Whitney tests",
      "- [x] Figures fig1-fig4", "- [ ] LaTeX manuscript -- next conversation",
      "", "---", "", "*Generated by p4_pipeline.ipynb*"]

report = "\n".join(L)
with open(os.path.join(RESULTS_DIR, "claude_report.md"), "w") as f:
    f.write(report + "\n")

payload = {
    "generated_at": now, "solar_source": SOLAR_SOURCE, "star_source": STAR_SOURCE,
    "use_synthetic": USE_SYNTHETIC, "S_SUN_MEAN": S_SUN_MEAN,
    "solar": {k: solar_res.get(k) for k in
              ["H_full", "H_resid", "H_decades", "tau_ar1_minimum",
               "p_ar1_minimum", "beta", "beta_err", "beta_r2", "H_from_psd"]},
    "solar_ews": {k: {kk: vv for kk, vv in v.items()
                      if not isinstance(vv, np.ndarray)}
                  for k, v in solar_res["ews"].items()},
    "stars": json.loads(df_stars.to_json(orient="records")),
    "mann_whitney": json.loads(df_mw.to_json(orient="records")),
}
with open(os.path.join(RESULTS_DIR, "claude_report.json"), "w") as f:
    json.dump(payload, f, indent=2)

df_stars.to_csv(os.path.join(RESULTS_DIR, "p4_star_results.csv"), index=False)
df_mw.to_csv(os.path.join(RESULTS_DIR, "p4_mannwhitney.csv"), index=False)

print(f"saved: {RESULTS_DIR}/claude_report.md")
print(f"saved: {RESULTS_DIR}/claude_report.json")
print("=" * 66)
print("PASTE results/claude_report.md INTO A NEW CLAUDE CONVERSATION")
print("=" * 66)
print()
print(report)

## 11 · Next steps

1. Set `USE_SYNTHETIC = False` and re-run with the real data.
2. Check the `CHECK` flags printed in §4.2 — any star whose measured `S_mean` differs
   from the Table 2 reference by more than 0.015 needs investigating before the numbers
   go into the manuscript.
3. Copy `results/claude_report.md` into a new Claude Projects conversation:

```
Paper P4 -- Critical Slowing Down. See paper_state.md in Project Knowledge.
Status: analysis complete with real data. Report below.

[paste results/claude_report.md]

Task for this conversation: write Section 1 (Introduction) in LaTeX, ~1 page,
Solar Physics (svjour3) format.
```